In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.decomposition import PCA

rfm = pd.read_csv('../data/processed/rfm_clustered.csv', index_col='Customer ID')
rfm_scaled = pd.read_csv('../data/processed/rfm_scaled.csv', index_col='Customer ID')

print(rfm_scaled.shape)


(5881, 3)


In [2]:
pca = PCA(n_components=3)
rfm_pca = pca.fit_transform(rfm_scaled)

print("Explained variance per component:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.3f} ({v*100:.1f}%)")
print(f"  Total: {pca.explained_variance_ratio_.sum():.3f} ({pca.explained_variance_ratio_.sum()*100:.1f}%)")


Explained variance per component:
  PC1: 0.759 (75.9%)
  PC2: 0.189 (18.9%)
  PC3: 0.051 (5.1%)
  Total: 1.000 (100.0%)


In [3]:
components_df = pd.DataFrame(
    pca.components_,
    columns=rfm_scaled.columns,
    index=[f'PC{i+1}' for i in range(3)]
).round(3)

print(components_df)


     Recency  Frequency  Monetary
PC1   -0.504      0.618     0.603
PC2    0.859      0.292     0.420
PC3    0.083      0.730    -0.678


PC1 explains 75.9% of variance, confirming that overall customer value (frequency + monetary) is the dominant axis of differentiation between customers.

In [4]:
viz_df = pd.DataFrame({
    'PC1': rfm_pca[:, 0],
    'PC2': rfm_pca[:, 1],
    'PC3': rfm_pca[:, 2],
    'Cluster': rfm['Persona']
})

fig = px.scatter(viz_df, x='PC1', y='PC2', color='Cluster',
                 title='Customer Segments in PCA Space',
                 labels={'PC1': 'PC1 — Customer Value', 'PC2': 'PC2 — Recency'},
                 opacity=0.6)
fig.show()


In [5]:
fig3d = px.scatter_3d(viz_df, x='PC1', y='PC2', z='PC3', color='Cluster',
                      title='3D Customer Segments',
                      opacity=0.5)
fig3d.show()


The PCA scatter plot visually confirms the four segments are well-separated, particularly along PC1 (Customer Value). Champions cluster distinctly at the far right, while Lost customers occupy the far left with high PC2 values reflecting their long absence. New/Promising and At-Risk segments overlap slightly in the middle, consistent with their similar moderate RFM profiles.